In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import data_loaders

data = data_loaders.DataLoader()
datasets = {
    'price': data.day_ahead_price_data.copy(),
    'inflow': data.inflow_data.copy(),
    'volume': data.volume_data.copy(),
    'wv': data.synthetic_water_value_data.copy(),
    'min_flow': data.min_flow_constraint_data.copy(),
    'min_vol': data.min_volume_constraint_data.copy(),
    'target': data.commitment_decision_data.copy()
}

dfs = []
for prefix, df in datasets.items():
    if 'date' in df.columns:
        df = df.set_index('date')
    
    # Ensure DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
        
    # Strip timezone
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
        
    # Prefix columns (except target) to ensure unique column names
    if prefix != 'target':
        df = df.add_prefix(f"{prefix}_")
        
    dfs.append(df)

df_all = pd.concat(dfs, axis=1).ffill().bfill().dropna()

target_col = data.commitment_decision_data.columns[0]
X = df_all.drop(columns=data.commitment_decision_data.columns)
y = df_all[target_col]

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

clf = RandomForestClassifier()
clf.fit(X_train, y_train)
accuracy = clf.score(X_test, y_test)
print(f"Accuracy for {target_col}: {accuracy:.4f}")

C:\Users\gebruiker\AppData\Local\Temp\ipykernel_14420\4086071309.py:35: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_all = pd.concat(dfs, axis=1).ffill().bfill().dropna()


Accuracy for Run No: 0.9994
